# 04 Route Accessibility

This notebook links each residential address to one school.

The idea is: 

1. each address already belongs to a school district
2. some districts have more than one school
3. in those cases, each address is assigned to the nearest school inside its own district.

We use straight-line euclidean distrance for this allocation step.

Later, when we compute actual routes, we will use the network instead to measure distance.

In [1]:
import pandas as pd
import geopandas as gpd

## Load the datasets

Here we load the two layers we need for the allocation step:

- the address layer
- the school layer already matched to the school districts

At this stage, we do not need the stret network yet.

In [2]:
addresses = gpd.read_file(
    "../data/processed/core_layers.gpkg",
    layer="district_addresses"
)

schools_in_districts = gpd.read_file(
    "../data/processed/core_layers.gpkg",
    layer="schools_in_districts"
)

## Check the structure of the datasets

Before doing anything else, we take a quick look at the columns.

This is useful because the school name column may not always have the same name, dependingon how the layer was saved ealier in the project.

In [3]:
print(addresses.columns.tolist())
print("_______________________________________________")
print(schools_in_districts.columns.tolist())

['Adresse', 'Vejnavn', 'Husnummer', 'Postnr', 'Bydelsnavn', 'Skoledistriktsnr', 'Skoledistriktsnavn', 'GIS_koor_X', 'GIS_koor_Y', 'geometry']
_______________________________________________
['school_name', 'school_type', 'index_right', 'temakode', 'temanavn', 'objekt_id', 'versions_i', 'systid_fra', 'systid_til', 'oprettet', 'cvr_kode', 'cvr_navn', 'kommunekod', 'bruger_id', 'oprindkode', 'oprindelse', 'statuskode', 'status', 'off_kode', 'offentlig', 'noegle', 'note', 'udd_distri', 'udd_dist_1', 'udd_dist_2', 'udd_dist_3', 'starttrin_', 'starttrin', 'sluttrin_k', 'slutttrin', 'sagsnr', 'link', 'udd_hovedo', 'udd_hove_1', 'udd_delomr', 'udd_delo_1', 'geometry']


##  Make the district codes match

The address layer and the school payer both contain school district codes.

To make sure they can be compared and merged safely, we convert both columns to text.

In [4]:
addresses["Skoledistriktsnr"] = addresses["Skoledistriktsnr"].astype(str)
schools_in_districts["udd_distri"] = schools_in_districts["udd_distri"].astype(str)

## Pick the school name column

Depending on the version of the data, the school name column may be called either "school_name" or "navn".

This small check makes the notebook work with either version.

In [5]:
school_name_col = "school_name" if "school_name" in schools_in_districts.columns else "navn"
print("Using school name column:", school_name_col)

Using school name column: school_name


## Get rid of unnessesary coloumns

The raw layers contain many columns, but for the allocation step we only need:

- an address identifier
- the district code and district name
- the school name
- the geometries

Keeping only the useful columns makes the rest of the notebook easier to read.

In [6]:
addresses = addresses[
    ["Adresse", "Skoledistriktsnr", "Skoledistriktsnavn", "geometry"]
].copy()

schools_in_districts = schools_in_districts[
    [school_name_col, "udd_distri", "udd_dist_1", "geometry"]
].copy()

## Rename columns to make them easier to work with

The original column names come from different datasets and use different naming styles.

Here we rename them so both layers use the same simple names.

In [7]:
addresses = addresses.rename(columns={
    "Skoledistriktsnr": "district_code",
    "Skoledistriktsnavn": "district_name_address",
    "geometry": "address_geometry"
})

schools_in_districts = schools_in_districts.rename(columns={
    school_name_col: "school_name",
    "udd_distri": "district_code",
    "udd_dist_1": "district_name_school",
    "geometry": "school_geometry"
})

## Turn the cleaned tables back into GeoDataFrames

Becauyse we renamed the geometry columns, we now tell GeoPandas again which geometry columns belongs to each dataset.

In [8]:
addresses = gpd.GeoDataFrame(
    addresses,
    geometry="address_geometry",
    crs="EPSG:25832"
)

schools_in_districts = gpd.GeoDataFrame(
    schools_in_districts,
    geometry="school_geometry",
    crs="EPSG:25832"
)

## Add an ID for each address

We give each address its own ID.

This makes it easier to keep track of addresses later when one address is matched to multiple candidate schools before we pick the nearest one.

In [9]:
addresses = addresses.reset_index(drop=True)
addresses["address_id"] = addresses.index

## Check that district codes line up

Before matching addresses to schools, we make sure the district codes still match between the two layers.

If both differences are empty, then the distrct codes line up correctly.

In [10]:
addr_ids = set(addresses["district_code"])
school_ids = set(schools_in_districts["district_code"])

print("Only in addresses:")
print(sorted(addr_ids - school_ids))

print("\nOnly in schools:")
print(sorted(school_ids - addr_ids))

Only in addresses:
[]

Only in schools:
[]


## Check how many schools each district has

Some districts only have one school, while others have more than one.

This matters because addresses in multi.chool districts will have several possible school matches before we choose the nearest one.

In [11]:
schools_per_district = (
    schools_in_districts.groupby("district_code")
    .size()
    .sort_values(ascending=False)
)

print(schools_per_district.head(20))

district_code
281474    4
101174    3
101062    2
280425    2
101035    2
101022    2
101074    2
101041    2
101015    2
101064    2
101063    2
101007    2
281980    2
101069    1
101157    1
101075    1
101076    1
101138    1
101151    1
101070    1
dtype: int64


## Match each address to all schools in the same district

This merge does not choose a final school yet. It only created the candidate list, meaning that each address is mathed to every school inside its own district.

In [12]:
address_school_candidates = addresses.merge(
    schools_in_districts,
    on="district_code",
    how="left"
)

## See how many candidate schools each address has

This gives a quick overview of how many addresses had:

- 1 possible school
- 2 possible schools
- 3 possible schools
- 4 possible schools

In [13]:
candidates_per_address = (
    address_school_candidates.groupby("address_id")
    .size()
)

print(candidates_per_address.value_counts().sort_index())

1    64847
2    15515
3     1026
4     3663
Name: count, dtype: int64


## Compute euclidean distance to each candidate school

Now we calculate the straight-line distance from each address to each school candidate inside the same district.

The data is measured in meters.

In [14]:
address_school_candidates["euclidean_dist_m"] = (
    address_school_candidates["address_geometry"].distance(
        address_school_candidates["school_geometry"]
    )
)

## Keep only the nearest school for each address

At this point, addresses in multi-school districts still appear more than once. 

Here we keep only the row with the smallest euclidean distance, so that each address ends up with one final school assigned to it.

In [15]:
nearest_idx = (
    address_school_candidates.groupby("address_id")["euclidean_dist_m"]
    .idxmin()
)

address_allocations = address_school_candidates.loc[nearest_idx].copy()

## Check results to verify

These checks tell us whether the allocation worked as expected.

- the number of allocated rows should match the number of addresses
- each address should appear only once
- there should be no missing school asignments

In [16]:
print("Addresses:", len(addresses))
print("Allocated rows:", len(address_allocations))
print("Unique addresses allocated:", address_allocations["address_id"].nunique())
print("Missing school assignments:", address_allocations["school_name"].isna().sum())

Addresses: 85051
Allocated rows: 85051
Unique addresses allocated: 85051
Missing school assignments: 0


## A few examples of the allocated addresses

Here we inspect the first few rows of the final address-to-school allocation.

In [17]:
address_allocations[
    ["Adresse", "district_code", "district_name_address", "school_name", "euclidean_dist_m"]
].head(10)

,Adresse,district_code,district_name_address,school_name,euclidean_dist_m
0,A-Vej 13,281053,Nordøstamager Skole,Nordøstamager Skole,1800.112982
1,A.C. Meyers Vænge 1,281060,Sluseholmen Skole,Sluseholmen Skole,1028.066690
2,A.C. Meyers Vænge 2,281060,Sluseholmen Skole,Sluseholmen Skole,1073.328721
3,A.C. Meyers Vænge 3,281060,Sluseholmen Skole,Sluseholmen Skole,1018.977117
4,A.C. Meyers Vænge 4,281060,Sluseholmen Skole,Sluseholmen Skole,1083.809253
5,A.C. Meyers Vænge 5,281060,Sluseholmen Skole,Sluseholmen Skole,1008.818563
6,A.C. Meyers Vænge 6,281060,Sluseholmen Skole,Sluseholmen Skole,1095.516333
7,A.C. Meyers Vænge 8,281060,Sluseholmen Skole,Sluseholmen Skole,1056.178049
8,A.C. Meyers Vænge 9,281060,Sluseholmen Skole,Sluseholmen Skole,908.954628
9,A.C. Meyers Vænge 10,281060,Sluseholmen Skole,Sluseholmen Skole,1060.531591


## Prepare a clean version for saving

The allocation table currently contains both address ceometry and school geometry.

This is useful whole working in Python, but a GeoPackage layer can only store one active geometry column in a simple way.

So here we make a separate export version of the table:

- keep the address point as the geometry
- save the school location as x and y coordinates
- remove the extra school geometry column before saving

This way, the result can still be resued later for routing.

In [18]:
export_allocations = address_allocations.copy()

## Convert school geometry to a single point

Some school geometries are not simple point geometries. 

To make the data easier to work with and save, we convert each school geometry to a single point using its centroid.

We then store the school location as x and y coordinates.

In [19]:
export_allocations["school_point"] = export_allocations["school_geometry"].centroid

export_allocations["school_x"] = export_allocations["school_point"].x
export_allocations["school_y"] = export_allocations["school_point"].y

## Keep only one geometry column

Geopackage layers can only store one active geometry column.

Here we keep the address point as the geometry and remove the extra school geometry.

In [20]:
export_allocations = export_allocations.set_geometry("address_geometry")

export_allocations = export_allocations.drop(columns=[
    "school_geometry",
    "school_point"
])

## Check the cleaned table

Before saving we quickly check that the geometry column is correct

In [21]:
print(export_allocations.dtypes)
print()
print("Active geometry column:", export_allocations.geometry.name)

Adresse                    object
district_code              object
district_name_address      object
address_geometry         geometry
address_id                  int64
school_name                object
district_name_school       object
euclidean_dist_m          float64
school_x                  float64
school_y                  float64
dtype: object

Active geometry column: address_geometry


## Save the final allocation layer

Finally, we save the result so it can be reused later in the routing part of the project.

This layer contains one school assignment for each address.

In [22]:
export_allocations.to_file(
    "../data/processed/core_layers.gpkg",
    layer="address_school_allocations",
    driver="GPKG"
)